# CIL Training — Equipo 25
Conditional Imitation Learning (Codevilla 2017) — MR4010 Proyecto Final

In [ ]:
import sys, os
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = '/content/drive/MyDrive/MR4010_proyecto_final_2026'
else:
    BASE = os.path.dirname(os.path.dirname(os.path.abspath(__file__))) if '__file__' in dir() else \
           '/Users/joelbecerril/MNA_WORKSPACE/NAVEGACION_AUTONOMA/Semana_9/ActFinal/MR4010_proyecto_final_2026'

import os
# Usar dataset balanceado (menos rectos, modelo aprende curvas mejor)
CSV_PATH   = os.path.join(BASE, 'data', 'dataset_balanced.csv')
IMAGES_DIR = os.path.join(BASE, 'data', 'images')
MODEL_OUT  = os.path.join(BASE, 'models', 'cil_model_equipo25.pt')
os.makedirs(os.path.join(BASE, 'models'), exist_ok=True)
print('BASE:', BASE)
print('CSV:', CSV_PATH)


In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import csv, random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from collections import Counter

# device: CUDA > MPS (Apple Silicon) > CPU
if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
elif torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
else:
    DEVICE = torch.device('cpu')
print('Device:', DEVICE)


In [ ]:
# ── Exploración del dataset ───────────────────────────────────────────────────
rows = list(csv.DictReader(open(CSV_PATH)))
cmds = Counter(int(r['nav_command']) for r in rows)
total = len(rows)
labels = {0:'CONTINUE', 1:'RECTO', 2:'IZQUIERDA', 3:'DERECHA'}
print(f'Total: {total:,}')
for c in range(4):
    n = cmds[c]
    bar = '█' * int(n/total*40)
    print(f'  {labels[c]:10s}: {n:6,} ({n/total*100:5.1f}%) {bar}')


In [ ]:
# ── Dataset ───────────────────────────────────────────────────────────────────
IMG_W, IMG_H = 200, 88    # resolución paper Codevilla 2017
N_COMMANDS   = 4

AUGMENT = transforms.Compose([
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    transforms.GaussianBlur(3, sigma=(0.1, 1.0)),
])

NORMALIZE = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
])

class CILDataset(Dataset):
    def __init__(self, rows, base_dir, augment=False):
        self.rows    = rows
        self.base    = base_dir
        self.augment = augment

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        row   = self.rows[idx]
        cmd   = int(row['nav_command'])
        steer = float(row['steering_angle'])
        speed = float(row['speed_kmh']) / 50.0

        img_path = os.path.join(self.base, row['image_path'])
        img = Image.open(img_path).convert('RGB').resize((IMG_W, IMG_H))

        if self.augment and random.random() < 0.5:
            img   = img.transpose(Image.FLIP_LEFT_RIGHT)
            steer = -steer
            if cmd == 2: cmd = 3
            elif cmd == 3: cmd = 2

        if self.augment:
            img = AUGMENT(img)

        img_t   = NORMALIZE(img)
        speed_t = torch.tensor([speed], dtype=torch.float32)
        cmd_t   = torch.tensor(cmd, dtype=torch.long)
        steer_t = torch.tensor([steer], dtype=torch.float32)
        return img_t, speed_t, cmd_t, steer_t

# Split 85% train / 15% val estratificado por comando
random.seed(42)
by_cmd = {c: [r for r in rows if int(r['nav_command']) == c] for c in range(4)}
train_rows, val_rows = [], []
for c, rws in by_cmd.items():
    random.shuffle(rws)
    cut = int(len(rws) * 0.85)
    train_rows += rws[:cut]
    val_rows   += rws[cut:]

print(f'Train: {len(train_rows):,}  Val: {len(val_rows):,}')

train_ds = CILDataset(train_rows, BASE, augment=True)
val_ds   = CILDataset(val_rows,   BASE, augment=False)
# num_workers=0 y pin_memory=False: compatibles con MPS y notebooks
train_dl = DataLoader(train_ds, batch_size=64, shuffle=True,  num_workers=0, pin_memory=False)
val_dl   = DataLoader(val_ds,   batch_size=64, shuffle=False, num_workers=0, pin_memory=False)

In [ ]:
# ── Modelo CIL (Codevilla 2017) ───────────────────────────────────────────────
class CILModel(nn.Module):
    def __init__(self, n_commands=4):
        super().__init__()

        # CNN backbone
        # Input: 3 x 88 x 200
        # Después de convs + pools → 256 x 11 x 25
        # AdaptiveAvgPool2d(1,1) → 256 x 1 x 1 = 256 features (divisible en MPS)
        self.cnn = nn.Sequential(
            nn.Conv2d(3, 32, 5, stride=2, padding=2), nn.ReLU(),   # 32 x 44 x 100
            nn.Conv2d(32, 64, 3, stride=1, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),                                         # 64 x 22 x 50
            nn.Conv2d(64, 128, 3, stride=1, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),                                         # 128 x 11 x 25
            nn.Conv2d(128, 256, 3, stride=1, padding=1), nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1)),                            # 256 x 1 x 1
            nn.Flatten(),                                            # 256
            nn.Linear(256, 512), nn.ReLU(), nn.Dropout(0.2),
        )

        # Medición de velocidad
        self.speed_fc = nn.Sequential(
            nn.Linear(1, 128), nn.ReLU(),
            nn.Linear(128, 128), nn.ReLU(),
        )

        # Ramas por comando (Codevilla: una rama por acción)
        branch_input = 512 + 128
        self.branches = nn.ModuleList([
            nn.Sequential(
                nn.Linear(branch_input, 256), nn.ReLU(), nn.Dropout(0.2),
                nn.Linear(256, 256), nn.ReLU(),
                nn.Linear(256, 1), nn.Tanh(),
            )
            for _ in range(n_commands)
        ])

    def forward(self, img, speed, cmd):
        feat = self.cnn(img)
        spd  = self.speed_fc(speed)
        x    = torch.cat([feat, spd], dim=1)

        outs = torch.stack([b(x) for b in self.branches], dim=1)  # (B, 4, 1)
        idx  = cmd.view(-1, 1, 1).expand(-1, 1, 1)
        out  = outs.gather(1, idx).squeeze(1)                      # (B, 1)
        return out

model = CILModel(N_COMMANDS).to(DEVICE)
total_params = sum(p.numel() for p in model.parameters())
print(f'Parámetros: {total_params:,}')

In [ ]:
EPOCHS    = 40
LR        = 1e-4
MAX_STEER = 0.5
TOL_RAD   = 0.05

# Weighted MSE: penaliza más los errores en curvas (|steering| > 0.05)
def weighted_mse(pred, target):
    weight = 1.0 + 4.0 * (target.abs() > 0.1).float()  # 5x peso en curvas
    return (weight * (pred - target) ** 2).mean()

optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=15, gamma=0.5)

best_val_loss = float('inf')
history = {'train_mse': [], 'val_mse': [], 'val_mae': [], 'val_acc': []}

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss = 0.0
    for img, speed, cmd, steer in train_dl:
        img   = img.to(DEVICE)
        speed = speed.to(DEVICE)
        cmd   = cmd.to(DEVICE)
        steer_norm = (steer / MAX_STEER).clamp(-1, 1).to(DEVICE)

        optimizer.zero_grad()
        pred = model(img, speed, cmd)
        loss = weighted_mse(pred, steer_norm)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    train_loss /= len(train_dl)

    model.eval()
    val_mse, val_mae, val_acc = 0.0, 0.0, 0.0
    with torch.no_grad():
        for img, speed, cmd, steer in val_dl:
            img        = img.to(DEVICE)
            speed      = speed.to(DEVICE)
            cmd        = cmd.to(DEVICE)
            steer_norm = (steer / MAX_STEER).clamp(-1, 1).to(DEVICE)
            pred       = model(img, speed, cmd)

            val_mse  += torch.nn.functional.mse_loss(pred, steer_norm).item()
            pred_rad  = pred.cpu() * MAX_STEER
            error_abs = (pred_rad - steer).abs()
            val_mae  += error_abs.mean().item()
            val_acc  += (error_abs < TOL_RAD).float().mean().item()

    val_mse /= len(val_dl)
    val_mae /= len(val_dl)
    val_acc /= len(val_dl)

    history['train_mse'].append(train_loss)
    history['val_mse'].append(val_mse)
    history['val_mae'].append(val_mae)
    history['val_acc'].append(val_acc)
    scheduler.step()

    if val_mse < best_val_loss:
        best_val_loss = val_mse
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'val_mse': val_mse,
            'val_mae': val_mae,
            'val_acc': val_acc,
            'n_commands': N_COMMANDS,
            'img_size': (IMG_W, IMG_H),
            'max_steer': MAX_STEER,
        }, MODEL_OUT)
        star = ' ★'
    else:
        star = ''

    print(f'Epoch {epoch:3d}/{EPOCHS} | train={train_loss:.4f} | val_mse={val_mse:.4f} | mae={val_mae:.4f}rad | acc={val_acc*100:.1f}%{star}')

print(f'\nMejor val_mse: {best_val_loss:.4f}  →  {MODEL_OUT}')


In [ ]:
# ── Curvas de aprendizaje ─────────────────────────────────────────────────────
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

ax1.plot(history['train_mse'], label='Train MSE')
ax1.plot(history['val_mse'],   label='Val MSE')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('MSE Loss')
ax1.set_title('CIL — MSE Loss')
ax1.legend()

ax2.plot(history['val_mae'], label='Val MAE (rad)', color='orange')
ax2_r = ax2.twinx()
ax2_r.plot(history['val_acc'], label='Val Acc (%)', color='green')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('MAE (rad)')
ax2_r.set_ylabel('Accuracy (%)')
ax2.set_title('CIL — MAE & Accuracy')
ax2.legend(loc='upper left')
ax2_r.legend(loc='upper right')

plt.tight_layout()
plt.savefig(os.path.join(BASE, 'models', 'learning_curve.png'), dpi=120)
plt.show()
print(f"Mejor val_mse: {min(history['val_mse']):.5f}  (epoch {history['val_mse'].index(min(history['val_mse']))+1})")
print(f"Mejor val_mae: {min(history['val_mae']):.4f} rad")
print(f"Mejor val_acc: {max(history['val_acc'])*100:.1f}%")


In [ ]:
# ── Verificación rápida del modelo guardado ───────────────────────────────────
import torch, torch.nn as nn

ckpt = torch.load(MODEL_OUT, map_location='cpu')
print(f"Epoch guardado : {ckpt['epoch']}")
# Compatibilidad con ambas versiones del checkpoint (val_mse o val_loss)
vloss = ckpt.get('val_mse', ckpt.get('val_loss', '?'))
print(f"Val loss       : {vloss:.5f}" if isinstance(vloss, float) else f"Val loss: {vloss}")
print(f"Img size       : {ckpt['img_size']}")
print(f"Comandos       : {ckpt['n_commands']}")

model_test = CILModel(ckpt['n_commands'])
model_test.load_state_dict(ckpt['model_state_dict'])
model_test.eval()

labels = {0:'CONTINUE', 1:'RECTO', 2:'IZQUIERDA', 3:'DERECHA'}
dummy_img   = torch.zeros(1, 3, IMG_H, IMG_W)
dummy_speed = torch.tensor([[0.6]])

for cmd in range(4):
    dummy_cmd = torch.tensor([cmd])
    with torch.no_grad():
        pred = model_test(dummy_img, dummy_speed, dummy_cmd)
    steer_rad = float(pred[0, 0]) * MAX_STEER
    print(f"  CMD {cmd} ({labels[cmd]:10s}): steering = {steer_rad:+.4f} rad")

print("\nInferencia OK — modelo listo para autonomous_cil.py")
